# Multinomial Logit — Standalone Fit and Search

This notebook shows:
1. How to fit a Multinomial Logit (MNL) model directly
2. How to run the SA search over MNL specifications

The MNL is the simplest discrete choice model. Each decision-maker chooses
the alternative with the highest utility, where utility is a linear function
of observed attributes plus an i.i.d. Gumbel error term.

MLE is JAX-accelerated — gradients are computed via automatic differentiation.

In [ ]:
!pip install SearchLibrium --upgrade -q

In [ ]:
import numpy as np
import pandas as pd
from SearchLibrium import MultinomialLogit, Parameters, call_siman

## 1. Load data

We use the Travel Mode dataset (210 individuals, 4 alternatives: air, train, bus, car).

In [ ]:
url = 'https://raw.githubusercontent.com/zahern/HypothesisX/refs/heads/main/data/TravelMode.csv'
df  = pd.read_csv(url)
df['AV']     = 1
df['CHOICE'] = df['choice'].map({'no': 0, 'yes': 1})
print(df.head(8))
print('Shape:', df.shape)

## 2. Standalone MNL fit

Fit a fixed specification using all candidate variables.

In [ ]:
varnames   = ['gcost', 'wait', 'vcost', 'travel']
alts       = np.unique(df['mode'])
choice_set = alts.tolist()

X      = df[varnames].values
y      = df['CHOICE'].values
ids    = df['individual'].values
alts_v = df['mode'].values

mnl = MultinomialLogit()
mnl.setup(
    X        = X,
    y        = y,
    varnames = varnames,
    alts     = alts_v,
    ids      = ids,
    base_alt = 'air',
)
mnl.fit()
mnl.summarise()

## 3. Search over MNL specifications

The SA search will explore subsets of variables, optional Box-Cox
transformations, and report the best model by BIC.
Backward elimination is applied at every evaluation to ensure all
retained variables are statistically significant.

In [ ]:
params = Parameters(
    criterions   = [('bic', -1)],          # minimise BIC
    df           = df,
    varnames     = varnames,
    asvarnames   = varnames,
    isvarnames   = [],
    choice_set   = choice_set,
    choices      = df['CHOICE'].values,
    alt_var      = df['mode'].values,
    choice_id    = df['individual'].values,
    base_alt     = 'air',
    models       = ['multinomial'],
    allow_bcvars = True,                   # search over Box-Cox transforms
    p_val        = 0.05,
    all_sig      = True,
)

best = call_siman(params, init_sol=None, id_num=1,
                  ctrl=(200, 0.001, 50, 10))
print('\nBest solution:', best)

## 4. Inspect the best model

In [ ]:
if best and best.get('model'):
    best['model'].summarise()

print('Variables:', best.get('asvars'))
print('BIC:      ', best.get('bic'))
print('Log-lik:  ', best.get('loglik'))